# 1. Exploratory Data Analysis: Video Game Sales
**Objective:** Load the raw data, inspect its shape, check the data types, and view the initial rows.

In [33]:
import pandas as pd

# Load the raw dataset
raw_sales_data = pd.read_csv('vgsales.csv')

# 1. Display the shape (number of rows and columns)
print(f"Dataset Shape: {raw_sales_data.shape}\n")

# 2. Display the data types of each column
print("Data Types:")
print(raw_sales_data.dtypes)
print("\n")

# 3. Display the first 5 rows
raw_sales_data.head()

Dataset Shape: (16598, 11)

Data Types:
Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object




,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


## 2. Data Cleaning Pipeline
**Objective:** Hunt down missing values, handle them using an explicit strategy, remove any duplicate records, and create a pristine working copy of the dataset.

In [34]:
# Check for missing values (NaNs) in the dataset
print("--- Missing Values ---")
missing_data = raw_sales_data.isnull().sum()
print(missing_data[missing_data > 0]) # only show those columns which have missing data

# Check for identical duplicate rows
duplicate_count = raw_sales_data.duplicated().sum()
print(f"\nTotal duplicate rows: {duplicate_count}")

--- Missing Values ---
Year         271
Publisher     58
dtype: int64

Total duplicate rows: 0


In [35]:
# Create a fresh copy to avoid overwriting our raw data
clean_sales_data = raw_sales_data.copy()

# STRATEGY FOR MISSING VALUES:
# 1. 'Year': Since predicting a specific release year is highly inaccurate, we will drop rows where 'Year' is missing.
# 2. 'Publisher': We don't want to lose the valid sales data for these rows, so we will fill missing publishers with 'Unknown'.

# Apply the strategy
clean_sales_data = clean_sales_data.dropna(subset=['Year'])  # Drop rows where 'Year' is missing
clean_sales_data['Publisher'] = clean_sales_data['Publisher'].fillna('Unknown') # Fill missing 'Publisher' with 'Unknown'

# We will drop all duplicate rows to ensure the dataset is clean and each entry is unique.
# In this dataset, there are no duplicate rows, but this step is included for completeness.
clean_sales_data = clean_sales_data.drop_duplicates()

# Final Verification
print(f"New Dataset Shape: {clean_sales_data.shape}")
print(f"Remaining Missing Values: {clean_sales_data.isnull().sum().max()}") # will give 0 if cleaning went well

New Dataset Shape: (16327, 11)
Remaining Missing Values: 0


## 3. Data Aggregation & Business Insights
**Objective:** Group the data to discover which game genres and platforms generate the most global revenue.

In [36]:
# Group by Genre and calculate total global sales
genre_sales = clean_sales_data.groupby('Genre')['Global_Sales'].sum().reset_index()

# Sort the values to see the highest-grossing genres first
top_genres = genre_sales.sort_values(by='Global_Sales', ascending=False)

print("--- All Video Game Genres by Global Sales (Millions) ---")
top_genres

--- All Video Game Genres by Global Sales (Millions) ---


,Genre,Global_Sales
0,Action,1722.88
10,Sports,1309.24
8,Shooter,1026.20
7,Role-Playing,923.84
4,Platform,829.15
3,Misc,797.62
6,Racing,726.77
2,Fighting,444.05
9,Simulation,390.16
5,Puzzle,242.22


### Regional Revenue by Top Publishers
**Business Question:** Which game publishers dominate the global market, and is their total revenue driven more by North American or Japanese sales?

**Objective:** Group the data by Publisher and aggregate multiple regional sales columns to compare market dependencies.

In [37]:
# Group by Publisher and sum up sales for multiple specific regions
publisher_sales = clean_sales_data.groupby('Publisher')[['NA_Sales', 'JP_Sales', 'Global_Sales']].sum().reset_index()

# Sort the values to see the highest-grossing publishers first
top_publishers = publisher_sales.sort_values(by='Global_Sales', ascending=False)

print("--- Top 10 Game Publishers: Regional Breakdown (Millions) ---")
top_publishers.head(10)

--- Top 10 Game Publishers: Regional Breakdown (Millions) ---


,Publisher,NA_Sales,JP_Sales,Global_Sales
359,Nintendo,815.75,454.99,1784.43
138,Electronic Arts,584.22,13.98,1093.39
21,Activision,426.01,6.54,721.41
455,Sony Computer Entertainment,265.22,74.10,607.28
524,Ubisoft,252.81,7.33,473.54
493,Take-Two Interactive,220.47,5.83,399.30
487,THQ,208.60,5.01,340.44
275,Konami Digital Entertainment,88.91,90.93,278.56
445,Sega,108.78,56.19,270.70
347,Namco Bandai Games,69.38,126.84,253.65
